<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания: 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

<b> Описание задачи: </b>

Создать базовый класс PaymentMethod в C#, который будет представлять
различные способы оплаты. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.

<b> Требования к базовому классу PaymentMethod: </b>

<b> • Атрибуты: </b> ID способа оплаты (PaymentMethodId), Название способа оплаты
(MethodName), Минимальная сумма (MinAmount).

<b> • Методы: </b>

- ProcessPayment(decimal amount): метод для обработки платежа
указанной суммы.

- CheckMinimumAmount(decimal amount): метод для проверки
минимальной суммы платежа.

- GetPaymentDetails(): метод для получения деталей способа оплаты.

<b> Требования к производным классам: </b>

1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные
атрибуты, такие как URL платежной системы (PaymentUrl).
Метод ProcessPayment() должен быть переопределен для включения URL
платежной системы в процесс оплаты.
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные
атрибуты, такие как Банковские данные (BankData).
Метод CheckMinimumAmount() должен быть переопределен для проверки
минимальной суммы платежа с учетом банковских комиссий.
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать
дополнительные атрибуты, такие как Место выдачи наличных
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен
для отображения места выдачи наличных.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [13]:
using System;
using System.Collections.Generic;
using System.Linq;

public delegate void PaymentDelegate(string message);
public delegate void BalanceDelegate(decimal oldBalance, decimal newBalance);

public abstract class PaymentMethod
{
    public int PaymentMethodId { get; set; }
    public string MethodName { get; set; }
    public decimal MinAmount { get; set; }
    public decimal Balance { get; set; }
    public string Currency { get; set; }
    
    public string Provider { get; set; }
    public string Description { get; set; }
    public List<string> Tags { get; set; }
    
    public event BalanceDelegate BalanceChanged;
    public event PaymentDelegate PaymentProcessed;

    public PaymentMethod()
    {
        Currency = "RUB";
        Balance = 0;
        Tags = new List<string>();
    }

    public virtual void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Произведена транзакция используя {MethodName} в размере: {amount} ");
        var oldBalance = Balance;
        Balance -= amount;
        BalanceChanged?.Invoke(oldBalance, Balance);
        PaymentProcessed?.Invoke($"Успешный платеж {amount}");
    }

    public virtual bool CheckMinimumAmount(decimal amount)
    {
        if (amount >= MinAmount)
        {
            Console.WriteLine("Средств: Достаточно");
            return true;
        }
        else
        {
            Console.WriteLine("Средств: Недостаточно");
            return false;
        }
    }

    public virtual void GetPaymentDetails()
    {
        Console.WriteLine($"ID: {PaymentMethodId} \nСпособ оплаты: {MethodName} \nМинимальная сумма: {MinAmount} \nБаланс: {Balance}");
    }

    public void ProcessPaymentWithCheck(decimal amount)
    {
        if (CheckMinimumAmount(amount))
        {
            ProcessPayment(amount);
        }
        else
        {
            Console.WriteLine("Транзакция отменена");
        }
    }

    public virtual void AddTag(string tag)
    {
        Tags.Add(tag);
        Console.WriteLine($"Добавлен тег: {tag}");
    }

    public virtual void ReplenishBalance(decimal amount)
    {
        var oldBalance = Balance;
        Balance += amount;
        Console.WriteLine($"Баланс пополнен на {amount}. Текущий баланс: {Balance}");
        BalanceChanged?.Invoke(oldBalance, Balance);
    }
}

public class PaymentCollection<T> where T : PaymentMethod
{
    private List<T> _items = new List<T>();
    
    public event Action<T> ItemAdded;

    public void Add(T item)
    {
        _items.Add(item);
        ItemAdded?.Invoke(item);
        item.BalanceChanged += (old, current) => 
            Console.WriteLine($"Баланс изменен: {item.MethodName} {old} -> {current}");
    }

    public void ProcessAll(decimal amount)
    {
        _items.ForEach(item => item.ProcessPaymentWithCheck(amount));
    }

    public void ShowAll()
    {
        _items.ForEach(item => item.GetPaymentDetails());
    }
}

class OnlinePayment : PaymentMethod
{
    public string PaymentUrl { get; set; }
    public List<string> Endpoints { get; set; }
    
    public OnlinePayment(string paymenturl) : base()
    {
        PaymentUrl = paymenturl;
        this.MethodName = "Онлайн платёж";
        this.PaymentMethodId = 1;
        Endpoints = new List<string>();
    }
    
    public override void ProcessPayment(decimal amount)
    {
        base.ProcessPayment(amount);
        Console.WriteLine($"по URL: {PaymentUrl}");
    }

    public void AddEndpoint(string endpoint)
    {
        Endpoints.Add(endpoint);
        Console.WriteLine($"Добавлен endpoint: {endpoint}");
    }
}

class BankTransfer : PaymentMethod
{
    public decimal BankData { get; set; }
    public List<string> Beneficiaries { get; set; }
    
    public BankTransfer(decimal bankdata) : base()
    {
        BankData = bankdata; 
        this.MethodName = "Банковский перевод";     
        this.PaymentMethodId = 2;
        Beneficiaries = new List<string>();
    }
    
    public override bool CheckMinimumAmount(decimal amount)
    {
        if (amount >= MinAmount + BankData)
        {
            Console.WriteLine("Средств: Достаточно");
            return true;
        }
        else
        {
            Console.WriteLine("Средств: Недостаточно");
            return false;
        }
    }

    public void AddBeneficiary(string beneficiary)
    {
        Beneficiaries.Add(beneficiary);
        Console.WriteLine($"Добавлен получатель: {beneficiary}");
    }
}

class CashPayment : PaymentMethod
{
    public string CashPickupPoint { get; set; }
    public List<string> AvailableCurrencies { get; set; }
    
    public CashPayment(string cashpickuppoint) : base()
    {
        CashPickupPoint = cashpickuppoint;
        this.MethodName = "Наличный расчёт";
        this.PaymentMethodId = 3;
        AvailableCurrencies = new List<string> { "RUB", "USD" };
    }
    
    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Место выдачи наличных: {CashPickupPoint}");
    }

    public void AddCurrency(string currency)
    {
        AvailableCurrencies.Add(currency);
        Console.WriteLine($"Добавлена валюта: {currency}");
    }
}

{
    {
        string URL = "14857291098459839";
        int payment = 520;

        OnlinePayment onlcash = new OnlinePayment(URL);
        onlcash.MinAmount = 100;
        onlcash.ReplenishBalance(2000);
        onlcash.GetPaymentDetails();
        onlcash.ProcessPaymentWithCheck(payment); 
        onlcash.AddTag("интернет");
        onlcash.AddEndpoint("/api/v1/pay");

        Console.WriteLine("\n-----------------------------------------\n");

        BankTransfer bankcash = new BankTransfer(120);
        bankcash.MinAmount = 1000;
        bankcash.ReplenishBalance(5000);
        bankcash.GetPaymentDetails();
        bankcash.ProcessPaymentWithCheck(payment); 
        bankcash.AddBeneficiary("ИП Петров");

        Console.WriteLine("\n-----------------------------------------\n");

        CashPayment cash = new CashPayment("Сбербанк");
        cash.MinAmount = 150;
        cash.ReplenishBalance(3000);
        cash.GetPaymentDetails();
        cash.ProcessPaymentWithCheck(payment);
        cash.AddCurrency("EUR");

        Console.WriteLine("\n=== РАБОТА С КОЛЛЕКЦИЕЙ И СОБЫТИЯМИ ===\n");

        var collection = new PaymentCollection<PaymentMethod>();
        collection.ItemAdded += (item) => 
            Console.WriteLine($"Добавлен в коллекцию: {item.MethodName}");

        onlcash.BalanceChanged += (old, current) => 
            Console.WriteLine($"Онлайн баланс изменился: {old} -> {current}");

        collection.Add(onlcash);
        collection.Add(bankcash);
        collection.Add(cash);

        Console.WriteLine("\nОбработка всех платежей через коллекцию:");
        collection.ProcessAll(300);

        Console.WriteLine("\nИнформация о всех платежных методах:");
        collection.ShowAll();
    }
}

Баланс пополнен на 2000. Текущий баланс: 2000
ID: 1 
Способ оплаты: Онлайн платёж 
Минимальная сумма: 100 
Баланс: 2000
Средств: Достаточно
Произведена транзакция используя Онлайн платёж в размере: 520 
по URL: 14857291098459839
Добавлен тег: интернет
Добавлен endpoint: /api/v1/pay

-----------------------------------------

Баланс пополнен на 5000. Текущий баланс: 5000
ID: 2 
Способ оплаты: Банковский перевод 
Минимальная сумма: 1000 
Баланс: 5000
Средств: Недостаточно
Транзакция отменена
Добавлен получатель: ИП Петров

-----------------------------------------

Баланс пополнен на 3000. Текущий баланс: 3000
ID: 3 
Способ оплаты: Наличный расчёт 
Минимальная сумма: 150 
Баланс: 3000
Место выдачи наличных: Сбербанк
Средств: Достаточно
Произведена транзакция используя Наличный расчёт в размере: 520 
Добавлена валюта: EUR

=== РАБОТА С КОЛЛЕКЦИЕЙ И СОБЫТИЯМИ ===

Добавлен в коллекцию: Онлайн платёж
Добавлен в коллекцию: Банковский перевод
Добавлен в коллекцию: Наличный расчёт

Обработка в